In [3]:
!pip install --quiet \
    datasets \
    kaggle \
    huggingface_hub \
    lightgbm \
    mlflow \
    pandas \
    scikit-learn \
    joblib \
    networkx \
    loguru \
    xgboost \
    catboost \
    optuna \
    shap \
    pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 64.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.4 MB/s eta 0:00:00


In [4]:
# download_and_save.py
import os
from pathlib import Path
import json
import sys
import pandas as pd
from loguru import logger

OUT = Path(os.getenv("OUTPUT_DIR", "/kaggle/working/artifacts"))
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# 1) PaySim via kagglehub (preferred) or fallback search
def download_paysim():
    try:
        import kagglehub
        logger.info("Using kagglehub to download PaySim")
        path = kagglehub.dataset_download("ealaxi/paysim1")  # returns local path or dir
        logger.info(f"PAYSIM dataset downloaded to {path}")
        # find CSV(s) under path
        for p in Path(path).rglob("*.csv"):
            df = pd.read_csv(p, low_memory=False)
            df.to_parquet(RAW / "paysim.parquet", index=False)
            logger.info(f"Saved PaySim as parquet ({p.name})")
            break
    except Exception as e:
        logger.warning(f"kagglehub not available or failed: {e}. Trying Kaggle CLI / local input search.")
        # fallback: try Kaggle CLI dataset path or /kaggle/input
        cand = None
        for p in Path("/kaggle/input").rglob("*.csv"):
            if "paysim" in p.name.lower() or "paysim" in str(p).lower():
                cand = p
                break
        if cand:
            df = pd.read_csv(cand, low_memory=False)
            df.to_parquet(RAW / "paysim.parquet", index=False)
            logger.info(f"Saved PaySim from {cand}")
        else:
            logger.error("PaySim not found in /kaggle/input; set PAYSIM_PATH env var.")

# 2) CIFER via Hugging Face datasets
def download_cifer():
    try:
        from datasets import load_dataset
        logger.info("Loading CIFER via datasets.load_dataset('Durgesh111/Cifer-Fraud-Detection-Dataset-AF')")
        ds = load_dataset("Durgesh111/Cifer-Fraud-Detection-Dataset-AF", split="train")
        # save a sample + metadata schema
        df = ds.to_pandas()
        # keep a sampled subset (full 21M can be huge); save sample and schema
        sample = df.sample(n=min(len(df), 200000), random_state=42)
        sample.to_parquet(RAW / "cifer_sample.parquet", index=False)
        with open(RAW / "cifer_schema.json","w") as f:
            json.dump({k: str(v) for k,v in ds.features.items()}, f, indent=2)
        logger.info("Saved CIFER sample + schema")
    except Exception as e:
        logger.exception("Failed to load CIFER dataset: " + str(e))

# 3) Nigerian dataset via Hugging Face datasets
def download_nigerian():
    try:
        from datasets import load_dataset
        logger.info("Loading Nigerian dataset via datasets.load_dataset('electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset')")
        ds = load_dataset("electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset", split="train")
        df = ds.to_pandas()
        # save full if small enough, otherwise sample & schema
        if len(df) <= 2_000_000:
            df.to_parquet(RAW / "nigerian.parquet", index=False)
            logger.info("Saved full Nigerian dataset")
        else:
            df.sample(n=500000, random_state=42).to_parquet(RAW / "nigerian_sample.parquet", index=False)
            logger.info("Saved Nigerian sample (large dataset)")
        with open(RAW / "nigerian_schema.json","w") as f:
            json.dump({k: str(v) for k,v in ds.features.items()}, f, indent=2)
        logger.info("Saved Nigerian schema")
    except Exception as e:
        logger.exception("Failed to load Nigerian dataset: " + str(e))

if __name__ == "__main__":
    download_paysim()
    download_cifer()
    download_nigerian()
    logger.info(f"Raw artifacts saved to {RAW}")

2026-03-01 17:38:55.580 | INFO     | __main__:download_paysim:17 - Using kagglehub to download PaySim
2026-03-01 17:38:55.674 | INFO     | __main__:download_paysim:19 - PAYSIM dataset downloaded to /kaggle/input/datasets/ealaxi/paysim1
2026-03-01 17:39:16.415 | INFO     | __main__:download_paysim:24 - Saved PaySim as parquet (PS_20174392719_1491204439457_log.csv)
2026-03-01 17:39:18.337 | INFO     | __main__:download_cifer:45 - Loading CIFER via datasets.load_dataset('Durgesh111/Cifer-Fraud-Detection-Dataset-AF')


README.md: 0.00B [00:00, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-10(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-11(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

...

2026-03-01 17:40:39.322 | INFO     | __main__:download_cifer:54 - Saved CIFER sample + schema
2026-03-01 17:40:40.827 | INFO     | __main__:download_nigerian:62 - Loading Nigerian dataset via datasets.load_dataset('electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset')


README.md: 0.00B [00:00, ?B/s]

V2-nigerian-financial-transactions-and-f(…):   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000000 [00:00<?, ? examples/s]

2026-03-01 17:41:37.422 | INFO     | __main__:download_nigerian:71 - Saved Nigerian sample (large dataset)
2026-03-01 17:41:37.425 | INFO     | __main__:download_nigerian:74 - Saved Nigerian schema
2026-03-01 17:41:38.157 | INFO     | __main__:<cell line: 0>:82 - Raw artifacts saved to /kaggle/working/artifacts/raw


In [11]:
# ============================================================
# BEHAVIORAL FRAUD DETECTION - PRODUCTION GRADE (FIXED)
# Uses Nigerian Dataset with Advanced Behavioral Analytics
# ============================================================

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

class BehavioralFraudDetector:
    """
    Advanced Behavioral Fraud Detection using Nigerian Transaction Dataset
    Focuses on user behavioral patterns, device consistency, temporal anomalies
    """
    
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()  # Better for outliers than StandardScaler
        self.encoders = {}
        self.feature_columns = []
        self.feature_importance = None
        
    def load_and_clean_data(self, data_path="/kaggle/working/artifacts/raw/nigerian_sample.parquet"):
        """Load and clean Nigerian dataset"""
        print("📊 Loading Nigerian dataset...")
        
        try:
            df = pd.read_parquet(data_path)
        except:
            # Fallback for different file formats
            df = pd.read_csv(data_path.replace('.parquet', '.csv'))
        
        print(f"   Original shape: {df.shape}")
        
        # Clean the data - handle infinite and extreme values
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        
        for col in numeric_columns:
            # Replace infinite values with NaN
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            
            # Cap extreme values at 99.9th percentile
            if df[col].notna().sum() > 0:
                cap_value = df[col].quantile(0.999)
                df[col] = df[col].clip(upper=cap_value)
        
        # Fill NaN values with reasonable defaults
        df = df.fillna(0)
        
        print(f"   Cleaned shape: {df.shape}")
        
        # Check if we have fraud labels
        if 'is_fraud' in df.columns:
            fraud_rate = df['is_fraud'].mean()
        elif 'isFraud' in df.columns:
            df['is_fraud'] = df['isFraud']
            fraud_rate = df['is_fraud'].mean()
        else:
            # Create synthetic fraud labels based on suspicious patterns
            df['is_fraud'] = self._create_synthetic_fraud_labels(df)
            fraud_rate = df['is_fraud'].mean()
        
        print(f"   Fraud rate: {fraud_rate:.3%}")
        return df
    
    def _create_synthetic_fraud_labels(self, df):
        """Create synthetic fraud labels for datasets without them"""
        print("🎯 Creating synthetic fraud labels...")
        
        # Define suspicious patterns
        suspicious_patterns = (
            (df.get('amount_ngn', df.get('amount', 0)) > df.get('amount_ngn', df.get('amount', 0)).quantile(0.99)) |  # Very large amounts
            (df.get('device_seen_count', 1) == 1) |  # New device
            (df.get('is_night_txn', False) == True) |  # Night transactions
            (df.get('velocity_score', 0) > 8) |  # High velocity
            (df.get('spending_deviation_score', 0) > 0.8)  # High spending deviation
        )
        
        return suspicious_patterns.astype(int)
    
    def create_behavioral_features(self, df):
        """Extract behavioral features from Nigerian dataset"""
        print("🧠 Engineering behavioral features...")
        
        features = df.copy()
        
        # ===== DEVICE BEHAVIOR PATTERNS =====
        print("   🖥️ Device behavioral features...")
        
        # Device trust and consistency
        if 'device_seen_count' in df.columns:
            features['device_trust_score'] = 1 / (df['device_seen_count'] + 1)
            features['device_newness'] = (df['device_seen_count'] <= 5).astype(int)
        else:
            features['device_trust_score'] = np.random.uniform(0.1, 1.0, len(df))
            features['device_newness'] = np.random.binomial(1, 0.2, len(df))
        
        if 'is_device_shared' in df.columns:
            features['device_exclusivity'] = (~df['is_device_shared']).astype(int)
        else:
            features['device_exclusivity'] = np.random.binomial(1, 0.7, len(df))
        
        # ===== TEMPORAL BEHAVIORAL PATTERNS =====
        print("   ⏰ Temporal behavioral features...")
        
        # Time-based behaviors
        if 'is_night_txn' in df.columns:
            features['night_risk'] = df['is_night_txn'].astype(int)
        else:
            features['night_risk'] = np.random.binomial(1, 0.15, len(df))
        
        if 'is_weekend' in df.columns:
            features['weekend_activity'] = df['is_weekend'].astype(int)
        else:
            features['weekend_activity'] = np.random.binomial(1, 0.3, len(df))
        
        if 'txn_hour' in df.columns:
            features['unusual_hour'] = ((df['txn_hour'] < 6) | (df['txn_hour'] > 22)).astype(int)
        else:
            features['unusual_hour'] = np.random.binomial(1, 0.25, len(df))
        
        # ===== VELOCITY AND FREQUENCY PATTERNS =====
        print("   🚀 Velocity behavioral features...")
        
        if 'velocity_score' in df.columns:
            features['velocity_risk'] = np.clip(df['velocity_score'] / 10, 0, 1)
        else:
            features['velocity_risk'] = np.random.uniform(0, 1, len(df))
        
        if 'user_txn_frequency_24h' in df.columns:
            features['burst_activity'] = (df['user_txn_frequency_24h'] > 10).astype(int)
            features['hyperactivity'] = (df['user_txn_frequency_24h'] > 50).astype(int)
        else:
            features['burst_activity'] = np.random.binomial(1, 0.1, len(df))
            features['hyperactivity'] = np.random.binomial(1, 0.02, len(df))
        
        # ===== SPENDING BEHAVIOR PATTERNS =====
        print("   💳 Spending behavioral features...")
        
        if 'spending_deviation_score' in df.columns:
            features['spending_anomaly'] = np.clip(df['spending_deviation_score'], 0, 1)
        else:
            features['spending_anomaly'] = np.random.uniform(0, 1, len(df))
        
        # Amount patterns
        amount_col = 'amount_ngn' if 'amount_ngn' in df.columns else 'amount'
        if amount_col in df.columns:
            features['amount_log'] = np.log1p(df[amount_col])
            features['round_amount'] = (df[amount_col] % 1000 == 0).astype(int)
            features['micro_transaction'] = (df[amount_col] < 100).astype(int)
            features['large_transaction'] = (df[amount_col] > df[amount_col].quantile(0.95)).astype(int)
        else:
            features['amount_log'] = np.random.uniform(0, 10, len(df))
            features['round_amount'] = np.random.binomial(1, 0.3, len(df))
            features['micro_transaction'] = np.random.binomial(1, 0.2, len(df))
            features['large_transaction'] = np.random.binomial(1, 0.05, len(df))
        
        # ===== NETWORK BEHAVIOR PATTERNS =====
        print("   🌐 Network behavioral features...")
        
        if 'is_ip_shared' in df.columns:
            features['ip_exclusivity'] = (~df['is_ip_shared']).astype(int)
        else:
            features['ip_exclusivity'] = np.random.binomial(1, 0.8, len(df))
        
        # Location and merchant patterns
        if 'location' in df.columns:
            location_counts = df['location'].value_counts()
            features['location_popularity'] = df['location'].map(location_counts) / len(df)
        else:
            features['location_popularity'] = np.random.uniform(0.001, 0.1, len(df))
        
        # ===== COMPOSITE BEHAVIORAL SCORES =====
        print("   🎯 Creating composite scores...")
        
        # Device behavior score
        features['device_behavior_score'] = (
            features['device_trust_score'] * 0.4 +
            features['device_exclusivity'] * 0.3 +
            (1 - features['device_newness']) * 0.3
        )
        
        # Temporal behavior score
        features['temporal_behavior_score'] = (
            (1 - features['night_risk']) * 0.4 +
            (1 - features['unusual_hour']) * 0.3 +
            (1 - features['weekend_activity']) * 0.3
        )
        
        # Transaction behavior score
        features['transaction_behavior_score'] = (
            (1 - features['velocity_risk']) * 0.3 +
            (1 - features['spending_anomaly']) * 0.3 +
            (1 - features['burst_activity']) * 0.2 +
            (1 - features['hyperactivity']) * 0.2
        )
        
        # Master behavioral risk score
        features['behavioral_risk_score'] = (
            (1 - features['device_behavior_score']) * 0.3 +
            (1 - features['temporal_behavior_score']) * 0.3 +
            (1 - features['transaction_behavior_score']) * 0.4
        )
        
        print(f"✅ Created {len([c for c in features.columns if c not in df.columns])} behavioral features")
        return features
    
    def select_features(self, df):
        """Select final behavioral feature set"""
        behavioral_features = [
            # Device behavior
            'device_trust_score', 'device_exclusivity', 'device_newness',
            
            # Temporal behavior
            'night_risk', 'weekend_activity', 'unusual_hour',
            
            # Velocity behavior
            'velocity_risk', 'burst_activity', 'hyperactivity',
            
            # Spending behavior
            'spending_anomaly', 'amount_log', 'round_amount', 
            'micro_transaction', 'large_transaction',
            
            # Network behavior
            'ip_exclusivity', 'location_popularity',
            
            # Composite scores
            'device_behavior_score', 'temporal_behavior_score', 
            'transaction_behavior_score', 'behavioral_risk_score'
        ]
        
        # Keep only existing features
        available_features = [f for f in behavioral_features if f in df.columns]
        print(f"📋 Selected {len(available_features)} behavioral features")
        
        return df[available_features]
    
    def train_model(self, data_path="/kaggle/working/artifacts/raw/nigerian_sample.parquet"):
        """Train behavioral fraud detection model"""
        print("🚀 TRAINING BEHAVIORAL FRAUD DETECTION MODEL")
        print("=" * 60)
        
        # Load and prepare data
        df = self.load_and_clean_data(data_path)
        df_features = self.create_behavioral_features(df)
        
        # Select features and target
        X = self.select_features(df_features)
        y = df['is_fraud'].astype(int)
        
        self.feature_columns = X.columns.tolist()
        print(f"🎯 Training with {len(self.feature_columns)} features")
        print(f"📊 Dataset: {len(X)} samples, fraud rate: {y.mean():.3%}")
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Scale features (RobustScaler handles outliers better)
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train model
        print("🎯 Training LightGBM model...")
        
        self.model = lgb.LGBMClassifier(
            objective='binary',
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight='balanced',
            verbose=-1
        )
        
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        auc_score = roc_auc_score(y_test, y_pred_proba)
        
        print(f"\n📈 BEHAVIORAL MODEL PERFORMANCE:")
        print(f"   AUC Score: {auc_score:.4f}")
        print("\n📋 Classification Report:")
        print(classification_report(y_test, y_pred))
        
        # Feature importance
        self.feature_importance = pd.DataFrame({
            'feature': self.feature_columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\n🔝 Top 10 Behavioral Features:")
        for idx, row in self.feature_importance.head(10).iterrows():
            print(f"   {row['feature']}: {row['importance']:.3f}")
        
        return auc_score
    
    def save_model(self, output_dir="/kaggle/working/models"):
        """Save model artifacts"""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Save model and preprocessing
        joblib.dump(self.model, output_path / "behavioral_model.pkl")
        joblib.dump(self.scaler, output_path / "behavioral_scaler.pkl")
        joblib.dump(self.feature_columns, output_path / "behavioral_features.pkl")
        
        if self.feature_importance is not None:
            self.feature_importance.to_csv(output_path / "behavioral_importance.csv", index=False)
        
        # Save metadata
        metadata = {
            "model_type": "behavioral_fraud_detection",
            "dataset": "nigerian_transactions",
            "features_count": len(self.feature_columns),
            "algorithm": "LightGBM"
        }
        
        with open(output_path / "behavioral_metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Behavioral model saved to {output_path}")

def train_behavioral_model():
    """Main training function"""
    detector = BehavioralFraudDetector()
    auc_score = detector.train_model()
    detector.save_model()
    print(f"\n🎉 Behavioral Model Complete! AUC: {auc_score:.4f}")

if __name__ == "__main__":
    train_behavioral_model()

🚀 TRAINING BEHAVIORAL FRAUD DETECTION MODEL
📊 Loading Nigerian dataset...
   Original shape: (500000, 45)
   Cleaned shape: (500000, 45)
   Fraud rate: 3.615%
🧠 Engineering behavioral features...
   🖥️ Device behavioral features...
   ⏰ Temporal behavioral features...
   🚀 Velocity behavioral features...
   💳 Spending behavioral features...
   🌐 Network behavioral features...
   🎯 Creating composite scores...
✅ Created 20 behavioral features
📋 Selected 20 behavioral features
🎯 Training with 20 features
📊 Dataset: 500000 samples, fraud rate: 3.615%
🎯 Training LightGBM model...

📈 BEHAVIORAL MODEL PERFORMANCE:
   AUC Score: 0.5029

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.58      0.73     96385
           1       0.04      0.42      0.07      3615

    accuracy                           0.58    100000
   macro avg       0.50      0.50      0.40    100000
weighted avg       0.93      0.58      0.70    100000


🔝 Top 10 

In [14]:
# ============================================================
# FINANCIAL CREDIT RISK MODEL - PRODUCTION GRADE
# Uses PaySim Dataset for Financial Transaction Analysis
# ============================================================

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

class FinancialCreditRiskModel:
    """
    Advanced Financial Credit Risk Model using PaySim Dataset
    Focuses on transaction amounts, account balances, payment patterns
    """
    
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()
        self.encoders = {}
        self.feature_columns = []
        self.feature_importance = None
    
    def load_and_clean_data(self, data_path="/kaggle/working/artifacts/raw/paysim.parquet"):
        """Load and clean PaySim dataset"""
        print("💰 Loading PaySim dataset...")
        
        try:
            df = pd.read_parquet(data_path)
        except:
            df = pd.read_csv(data_path.replace('.parquet', '.csv'))
        
        print(f"   Original shape: {df.shape}")
        
        # Clean infinite and extreme values
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            if df[col].notna().sum() > 0:
                cap_value = df[col].quantile(0.999)
                df[col] = df[col].clip(upper=cap_value)
        
        df = df.fillna(0)
        
        # Ensure we have fraud labels
        if 'isFraud' in df.columns:
            df['is_fraud'] = df['isFraud']
        elif 'is_fraud' not in df.columns:
            df['is_fraud'] = self._create_financial_fraud_labels(df)
        
        fraud_rate = df['is_fraud'].mean()
        print(f"   Fraud rate: {fraud_rate:.3%}")
        return df
    
    def _create_financial_fraud_labels(self, df):
        """Create financial fraud labels based on suspicious patterns"""
        print("🎯 Creating financial fraud labels...")
        
        # Financial fraud patterns
        suspicious = (
            (df['amount'] > df['amount'].quantile(0.98)) |  # Very large amounts
            ((df['type'] == 'TRANSFER') & (df['amount'] > 200000)) |  # Large transfers
            ((df['type'] == 'CASH_OUT') & (df['amount'] > 100000)) |  # Large cash outs
            (df['oldbalanceOrg'] == 0) |  # Empty origin accounts
            ((df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0) & (df['amount'] < df['oldbalanceOrg']))  # Balance inconsistencies
        )
        
        return suspicious.astype(int)
    
    def create_financial_features(self, df):
        """Extract comprehensive financial features"""
        print("💳 Engineering financial features...")
        
        features = df.copy()
        
        # ===== AMOUNT-BASED FEATURES =====
        print("   💰 Amount-based features...")
        
        features['amount_log'] = np.log1p(df['amount'])
        features['amount_sqrt'] = np.sqrt(df['amount'])
        features['is_round_amount'] = (df['amount'] % 1000 == 0).astype(int)
        features['is_large_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)
        features['is_micro_amount'] = (df['amount'] < 100).astype(int)
        
        # Amount patterns by transaction type
        features['amount_zscore_by_type'] = df.groupby('type')['amount'].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-6)
        )
        
        # ===== BALANCE-BASED FEATURES =====
        print("   🏦 Balance-based features...")
        
        # Balance ratios and changes
        features['balance_orig_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
        features['balance_dest_ratio'] = df['amount'] / (df['oldbalanceDest'] + 1)
        
        features['balance_orig_change'] = df['newbalanceOrig'] - df['oldbalanceOrg']
        features['balance_dest_change'] = df['newbalanceDest'] - df['oldbalanceDest']
        
        # Balance consistency checks
        features['balance_inconsistency'] = (
            abs(features['balance_orig_change'] + df['amount']) > 0.01
        ).astype(int)
        
        features['zero_balance_orig'] = (df['oldbalanceOrg'] == 0).astype(int)
        features['zero_balance_dest'] = (df['oldbalanceDest'] == 0).astype(int)
        
        # ===== TRANSACTION TYPE FEATURES =====
        print("   🔄 Transaction type features...")
        
        # Encode transaction types
        type_encoder = LabelEncoder()
        features['type_encoded'] = type_encoder.fit_transform(df['type'])
        self.encoders['type'] = type_encoder
        
        # Risk by transaction type
        features['high_risk_type'] = df['type'].isin(['TRANSFER', 'CASH_OUT']).astype(int)
        features['payment_type'] = (df['type'] == 'PAYMENT').astype(int)
        features['debit_type'] = (df['type'] == 'DEBIT').astype(int)
        
        # ===== CUSTOMER BEHAVIOR FEATURES =====
        print("   👤 Customer behavior features...")
        
        # Customer transaction patterns
        features['customer_txn_count'] = df.groupby('nameOrig')['amount'].transform('count')
        features['customer_total_amount'] = df.groupby('nameOrig')['amount'].transform('sum')
        features['customer_avg_amount'] = df.groupby('nameOrig')['amount'].transform('mean')
        features['customer_max_amount'] = df.groupby('nameOrig')['amount'].transform('max')
        features['customer_amount_std'] = df.groupby('nameOrig')['amount'].transform('std')
        
        # Customer risk indicators
        features['amount_vs_customer_avg'] = df['amount'] / (features['customer_avg_amount'] + 1)
        features['amount_vs_customer_max'] = df['amount'] / (features['customer_max_amount'] + 1)
        
        # Customer balance patterns
        features['customer_avg_balance'] = df.groupby('nameOrig')['oldbalanceOrg'].transform('mean')
        features['balance_vs_customer_avg'] = df['oldbalanceOrg'] / (features['customer_avg_balance'] + 1)
        
        # ===== DESTINATION ANALYSIS =====
        print("   🎯 Destination analysis features...")
        
        # Destination transaction patterns
        features['dest_txn_count'] = df.groupby('nameDest')['amount'].transform('count')
        features['dest_total_received'] = df.groupby('nameDest')['amount'].transform('sum')
        features['dest_avg_received'] = df.groupby('nameDest')['amount'].transform('mean')
        
        # Destination risk indicators
        features['popular_destination'] = (features['dest_txn_count'] > 100).astype(int)
        features['new_destination'] = (features['dest_txn_count'] <= 5).astype(int)
        
        # ===== TEMPORAL FINANCIAL FEATURES =====
        print("   ⏰ Temporal financial features...")
        
        # Time-based patterns
        features['transaction_hour'] = df['step'] % 24
        features['transaction_day'] = df['step'] // 24
        features['is_business_hours'] = ((features['transaction_hour'] >= 9) & 
                                       (features['transaction_hour'] <= 17)).astype(int)
        
        # ===== FINANCIAL RISK SCORES =====
        print("   ⚠️ Financial risk scoring...")
        
        # Amount risk score
        features['amount_risk_score'] = (
            features['is_large_amount'] * 0.3 +
            features['is_micro_amount'] * 0.2 +
            (abs(features['amount_zscore_by_type']) > 3).astype(int) * 0.5
        )
        
        # Balance risk score
        features['balance_risk_score'] = (
            features['balance_inconsistency'] * 0.4 +
            features['zero_balance_orig'] * 0.3 +
            (features['balance_orig_ratio'] > 10).astype(int) * 0.3
        )
        
        # Customer risk score
        features['customer_risk_score'] = (
            (features['customer_txn_count'] == 1).astype(int) * 0.3 +  # Single transaction
            (features['amount_vs_customer_avg'] > 5).astype(int) * 0.4 +  # Unusual amount
            (features['customer_amount_std'] > features['customer_avg_amount']).astype(int) * 0.3  # High variance
        )
        
        # Master financial risk score
        features['financial_risk_score'] = (
            features['amount_risk_score'] * 0.3 +
            features['balance_risk_score'] * 0.4 +
            features['customer_risk_score'] * 0.3
        )
        
        print(f"✅ Created {len([c for c in features.columns if c not in df.columns])} financial features")
        return features
    
    def select_features(self, df):
        """Select final financial feature set"""
        financial_features = [
            # Amount features
            'amount', 'amount_log', 'amount_sqrt', 'is_round_amount', 
            'is_large_amount', 'is_micro_amount', 'amount_zscore_by_type',
            
            # Balance features
            'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
            'balance_orig_ratio', 'balance_dest_ratio', 'balance_orig_change', 
            'balance_dest_change', 'balance_inconsistency', 'zero_balance_orig', 'zero_balance_dest',
            
            # Transaction type features
            'type_encoded', 'high_risk_type', 'payment_type', 'debit_type',
            
            # Customer features
            'customer_txn_count', 'customer_total_amount', 'customer_avg_amount',
            'customer_max_amount', 'customer_amount_std', 'amount_vs_customer_avg',
            'amount_vs_customer_max', 'customer_avg_balance', 'balance_vs_customer_avg',
            
            # Destination features
            'dest_txn_count', 'dest_total_received', 'dest_avg_received',
            'popular_destination', 'new_destination',
            
            # Temporal features
            'transaction_hour', 'is_business_hours',
            
            # Risk scores
            'amount_risk_score', 'balance_risk_score', 'customer_risk_score', 'financial_risk_score'
        ]
        
        # Keep only existing features
        available_features = [f for f in financial_features if f in df.columns]
        print(f"📋 Selected {len(available_features)} financial features")
        
        return df[available_features]
    
    def train_model(self, data_path="/kaggle/working/artifacts/raw/paysim.parquet"):
        """Train financial credit risk model"""
        print("🚀 TRAINING FINANCIAL CREDIT RISK MODEL")
        print("=" * 60)
        
        # Load and prepare data
        df = self.load_and_clean_data(data_path)
        df_features = self.create_financial_features(df)
        
        # Select features and target
        X = self.select_features(df_features)
        y = df['is_fraud'].astype(int)
        
        self.feature_columns = X.columns.tolist()
        print(f"🎯 Training with {len(self.feature_columns)} features")
        print(f"📊 Dataset: {len(X)} samples, fraud rate: {y.mean():.3%}")
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train model
        print("🎯 Training LightGBM model...")
        
        self.model = lgb.LGBMClassifier(
            objective='binary',
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight='balanced',
            verbose=-1
        )
        
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        auc_score = roc_auc_score(y_test, y_pred_proba)
        
        print(f"\n📈 FINANCIAL MODEL PERFORMANCE:")
        print(f"   AUC Score: {auc_score:.4f}")
        print("\n📋 Classification Report:")
        print(classification_report(y_test, y_pred))
        
        # Feature importance
        self.feature_importance = pd.DataFrame({
            'feature': self.feature_columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\n🔝 Top 10 Financial Features:")
        for idx, row in self.feature_importance.head(10).iterrows():
            print(f"   {row['feature']}: {row['importance']:.3f}")
        
        return auc_score
    
    def save_model(self, output_dir="/kaggle/working/models"):
        """Save model artifacts"""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Save model and preprocessing
        joblib.dump(self.model, output_path / "financial_model.pkl")
        joblib.dump(self.scaler, output_path / "financial_scaler.pkl")
        joblib.dump(self.encoders, output_path / "financial_encoders.pkl")
        joblib.dump(self.feature_columns, output_path / "financial_features.pkl")
        
        if self.feature_importance is not None:
            self.feature_importance.to_csv(output_path / "financial_importance.csv", index=False)
        
        # Save metadata
        metadata = {
            "model_type": "financial_credit_risk",
            "dataset": "paysim_transactions",
            "features_count": len(self.feature_columns),
            "algorithm": "LightGBM"
        }
        
        with open(output_path / "financial_metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Financial model saved to {output_path}")

def train_financial_model():
    """Main training function"""
    model = FinancialCreditRiskModel()
    auc_score = model.train_model()
    model.save_model()
    print(f"\n🎉 Financial Model Complete! AUC: {auc_score:.4f}")

if __name__ == "__main__":
    train_financial_model()

🚀 TRAINING FINANCIAL CREDIT RISK MODEL
💰 Loading PaySim dataset...
   Original shape: (6362620, 11)
   Fraud rate: 0.129%
💳 Engineering financial features...
   💰 Amount-based features...
   🏦 Balance-based features...
   🔄 Transaction type features...
   👤 Customer behavior features...
   🎯 Destination analysis features...
   ⏰ Temporal financial features...
   ⚠️ Financial risk scoring...
✅ Created 38 financial features
📋 Selected 42 financial features
🎯 Training with 42 features
📊 Dataset: 6362620 samples, fraud rate: 0.129%
🎯 Training LightGBM model...

📈 FINANCIAL MODEL PERFORMANCE:
   AUC Score: 0.9998

📋 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524


🔝 Top 10 Financial Features:
 

In [15]:
# ============================================================
# NETWORK SECURITY INTELLIGENCE - PRODUCTION GRADE
# Uses CIFER Dataset for Network Attack Detection
# ============================================================

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

class NetworkSecurityIntelligence:
    """
    Advanced Network Security Intelligence using CIFER Dataset
    Focuses on network attack patterns, bot detection, anomalous connections
    """
    
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()
        self.encoders = {}
        self.feature_columns = []
        self.feature_importance = None
    
    def load_and_clean_data(self, data_path="/kaggle/working/artifacts/raw/cifer_sample.parquet"):
        """Load and clean CIFER dataset"""
        print("🔒 Loading CIFER dataset...")
        
        try:
            df = pd.read_parquet(data_path)
        except:
            df = pd.read_csv(data_path.replace('.parquet', '.csv'))
        
        print(f"   Original shape: {df.shape}")
        
        # Clean data
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            if df[col].notna().sum() > 0:
                cap_value = df[col].quantile(0.999)
                df[col] = df[col].clip(upper=cap_value)
        
        df = df.fillna(0)
        
        # Create network security threat labels
        if 'is_attack' not in df.columns and 'isFraud' not in df.columns:
            df['is_attack'] = self._create_security_threat_labels(df)
        elif 'isFraud' in df.columns:
            df['is_attack'] = df['isFraud']
        
        threat_rate = df['is_attack'].mean()
        print(f"   Threat rate: {threat_rate:.3%}")
        return df
    
    def _create_security_threat_labels(self, df):
        """Create network security threat labels"""
        print("🎯 Creating network threat labels...")
        
        # Network security threat patterns
        threats = (
            (df['amount'] < 1).astype(int) |  # Micro transactions (card testing)
            (df.get('step', 0) % 100 < 5).astype(int) |  # Regular intervals (bot activity)
            ((df['type'] == 'TRANSFER') & (df['amount'] == df['amount'].round())).astype(int) |  # Exact amounts
            (df['oldbalanceOrg'] == 0).astype(int) |  # Suspicious account states
            ((df['amount'] > 100000) & (df['type'] == 'CASH_OUT')).astype(int)  # Large cash withdrawals
        )
        
        return threats.astype(int)
    
    def create_network_security_features(self, df):
        """Extract network security features"""
        print("🛡️ Engineering network security features...")
        
        features = df.copy()
        
        # ===== BOT DETECTION FEATURES =====
        print("   🤖 Bot detection features...")
        
        # Transaction regularity (bots have regular patterns)
        features['transaction_regularity'] = df.groupby('nameOrig')['step'].transform(
            lambda x: 1 / (np.std(np.diff(sorted(x))) + 1) if len(x) > 1 else 0
        )
        
        # Exact amount repetition (bot characteristic)
        features['exact_amount_repetition'] = df.groupby('nameOrig')['amount'].transform(
            lambda x: 1 - (x.nunique() / len(x))
        )
        
        # High-frequency activity
        features['high_frequency_user'] = df.groupby('nameOrig')['amount'].transform('count') > 20
        features['high_frequency_user'] = features['high_frequency_user'].astype(int)
        
        # ===== ATTACK PATTERN FEATURES =====
        print("   ⚔️ Attack pattern features...")
        
        # Micro transaction attacks (card testing)
        features['micro_transaction_attack'] = (df['amount'] < 1).astype(int)
        features['penny_transaction'] = (df['amount'] == 0.01).astype(int)
        
        # Velocity attacks
        features['velocity_attack'] = df.groupby('nameOrig')['step'].transform(
            lambda x: (len(x) / (x.max() - x.min() + 1) > 1) if len(x) > 1 else False
        ).astype(int)
        
        # Amount pattern attacks
        features['round_amount_pattern'] = (df['amount'] % 100 == 0).astype(int)
        features['sequential_pattern'] = df.groupby('nameOrig')['amount'].transform(
            lambda x: (x.diff().abs() < 10).sum() / len(x) if len(x) > 1 else 0
        )
        
        # ===== NETWORK ANOMALY FEATURES =====
        print("   🌐 Network anomaly features...")
        
        # Account creation patterns
        features['zero_balance_creation'] = (df['oldbalanceOrg'] == 0).astype(int)
        features['instant_large_transaction'] = (
            (df['oldbalanceOrg'] == 0) & (df['amount'] > 10000)
        ).astype(int)
        
        # Destination analysis
        features['popular_destination'] = df.groupby('nameDest')['amount'].transform('count') > 50
        features['popular_destination'] = features['popular_destination'].astype(int)
        
        features['new_destination'] = df.groupby('nameDest')['amount'].transform('count') <= 3
        features['new_destination'] = features['new_destination'].astype(int)
        
        # ===== TRANSACTION TYPE SECURITY FEATURES =====
        print("   🔄 Transaction type security features...")
        
        # Encode transaction types
        type_encoder = LabelEncoder()
        features['type_encoded'] = type_encoder.fit_transform(df['type'])
        self.encoders['type'] = type_encoder
        
        # Risky transaction types for network attacks
        features['transfer_risk'] = (df['type'] == 'TRANSFER').astype(int)
        features['cashout_risk'] = (df['type'] == 'CASH_OUT').astype(int)
        features['payment_pattern'] = (df['type'] == 'PAYMENT').astype(int)
        
        # ===== TEMPORAL ATTACK FEATURES =====
        print("   ⏰ Temporal attack features...")
        
        # Time-based attack patterns
        features['transaction_hour'] = df['step'] % 24
        features['night_attack'] = ((features['transaction_hour'] < 6) | 
                                  (features['transaction_hour'] > 22)).astype(int)
        
        # Weekend attacks
        features['weekend_attack'] = ((df['step'] // 24) % 7 >= 5).astype(int)
        
        # Burst patterns
        features['burst_pattern'] = df.groupby('nameOrig')['step'].transform(
            lambda x: (np.diff(sorted(x)) < 10).sum() / len(x) if len(x) > 1 else 0
        )
        
        # ===== BALANCE MANIPULATION FEATURES =====
        print("   💰 Balance manipulation features...")
        
        # Balance inconsistencies (potential manipulation)
        features['balance_inconsistency'] = (
            abs((df['oldbalanceOrg'] - df['amount']) - df['newbalanceOrig']) > 0.01
        ).astype(int)
        
        features['dest_balance_inconsistency'] = (
            abs((df['oldbalanceDest'] + df['amount']) - df['newbalanceDest']) > 0.01
        ).astype(int)
        
        # Unusual balance patterns
        features['emptied_account'] = (
            (df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0)
        ).astype(int)
        
        # ===== NETWORK SECURITY RISK SCORES =====
        print("   ⚠️ Network security risk scoring...")
        
        # Bot behavior score
        features['bot_behavior_score'] = (
            features['transaction_regularity'] * 0.3 +
            features['exact_amount_repetition'] * 0.3 +
            features['high_frequency_user'] * 0.4
        )
        
        # Attack pattern score
        features['attack_pattern_score'] = (
            features['micro_transaction_attack'] * 0.3 +
            features['velocity_attack'] * 0.3 +
            features['burst_pattern'] * 0.4
        )
        
        # Network anomaly score
        features['network_anomaly_score'] = (
            features['zero_balance_creation'] * 0.2 +
            features['instant_large_transaction'] * 0.4 +
            features['balance_inconsistency'] * 0.4
        )
        
        # Master security threat score
        features['security_threat_score'] = (
            features['bot_behavior_score'] * 0.3 +
            features['attack_pattern_score'] * 0.4 +
            features['network_anomaly_score'] * 0.3
        )
        
        print(f"✅ Created {len([c for c in features.columns if c not in df.columns])} security features")
        return features
    
    def select_features(self, df):
        """Select final network security feature set"""
        security_features = [
            # Bot detection
            'transaction_regularity', 'exact_amount_repetition', 'high_frequency_user',
            
            # Attack patterns
            'micro_transaction_attack', 'penny_transaction', 'velocity_attack',
            'round_amount_pattern', 'sequential_pattern',
            
            # Network anomalies
            'zero_balance_creation', 'instant_large_transaction', 
            'popular_destination', 'new_destination',
            
            # Transaction security
            'type_encoded', 'transfer_risk', 'cashout_risk', 'payment_pattern',
            
            # Temporal attacks
            'transaction_hour', 'night_attack', 'weekend_attack', 'burst_pattern',
            
            # Balance manipulation
            'balance_inconsistency', 'dest_balance_inconsistency', 'emptied_account',
            
            # Basic transaction features
            'amount', 'step', 'oldbalanceOrg', 'newbalanceOrig',
            
            # Risk scores
            'bot_behavior_score', 'attack_pattern_score', 'network_anomaly_score', 'security_threat_score'
        ]
        
        # Keep only existing features
        available_features = [f for f in security_features if f in df.columns]
        print(f"📋 Selected {len(available_features)} security features")
        
        return df[available_features]
    
    def train_model(self, data_path="/kaggle/working/artifacts/raw/cifer_sample.parquet"):
        """Train network security model"""
        print("🚀 TRAINING NETWORK SECURITY INTELLIGENCE MODEL")
        print("=" * 60)
        
        # Load and prepare data
        df = self.load_and_clean_data(data_path)
        df_features = self.create_network_security_features(df)
        
        # Select features and target
        X = self.select_features(df_features)
        y = df['is_attack'].astype(int)
        
        self.feature_columns = X.columns.tolist()
        print(f"🎯 Training with {len(self.feature_columns)} features")
        print(f"📊 Dataset: {len(X)} samples, threat rate: {y.mean():.3%}")
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train model
        print("🎯 Training LightGBM model...")
        
        self.model = lgb.LGBMClassifier(
            objective='binary',
            n_estimators=250,
            max_depth=7,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight='balanced',
            verbose=-1
        )
        
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        auc_score = roc_auc_score(y_test, y_pred_proba)
        
        print(f"\n📈 NETWORK SECURITY MODEL PERFORMANCE:")
        print(f"   AUC Score: {auc_score:.4f}")
        print("\n📋 Classification Report:")
        print(classification_report(y_test, y_pred))
        
        # Feature importance
        self.feature_importance = pd.DataFrame({
            'feature': self.feature_columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\n🔝 Top 10 Security Features:")
        for idx, row in self.feature_importance.head(10).iterrows():
            print(f"   {row['feature']}: {row['importance']:.3f}")
        
        return auc_score
    
    def save_model(self, output_dir="/kaggle/working/models"):
        """Save model artifacts"""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)
        
        # Save model and preprocessing
        joblib.dump(self.model, output_path / "security_model.pkl")
        joblib.dump(self.scaler, output_path / "security_scaler.pkl")
        joblib.dump(self.encoders, output_path / "security_encoders.pkl")
        joblib.dump(self.feature_columns, output_path / "security_features.pkl")
        
        if self.feature_importance is not None:
            self.feature_importance.to_csv(output_path / "security_importance.csv", index=False)
        
        # Save metadata
        metadata = {
            "model_type": "network_security_intelligence",
            "dataset": "cifer_network_data",
            "features_count": len(self.feature_columns),
            "algorithm": "LightGBM"
        }
        
        with open(output_path / "security_metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ Network security model saved to {output_path}")

def train_security_model():
    """Main training function"""
    model = NetworkSecurityIntelligence()
    auc_score = model.train_model()
    model.save_model()
    print(f"\n🎉 Security Model Complete! AUC: {auc_score:.4f}")

if __name__ == "__main__":
    train_security_model()

🚀 TRAINING NETWORK SECURITY INTELLIGENCE MODEL
🔒 Loading CIFER dataset...
   Original shape: (200000, 11)
   Threat rate: 0.130%
🛡️ Engineering network security features...
   🤖 Bot detection features...
   ⚔️ Attack pattern features...
   🌐 Network anomaly features...
   🔄 Transaction type security features...
   ⏰ Temporal attack features...
   💰 Balance manipulation features...
   ⚠️ Network security risk scoring...
✅ Created 27 security features
📋 Selected 31 security features
🎯 Training with 31 features
📊 Dataset: 200000 samples, threat rate: 0.130%
🎯 Training LightGBM model...

📈 NETWORK SECURITY MODEL PERFORMANCE:
   AUC Score: 0.5504

📋 Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     39948
           1       0.00      0.02      0.00        52

    accuracy                           0.98     40000
   macro avg       0.50      0.50      0.50     40000
weighted avg       1.00      0.98      0.99     4000

In [23]:
# ============================================================
# COMPREHENSIVE MODEL EVALUATION SYSTEM
# Evaluate all 3 models with detailed performance metrics
# ============================================================

import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

class ModelEvaluator:
    """Comprehensive evaluation system for all fraud detection models"""
    
    def __init__(self, artifacts_path="/kaggle/working/models"):
        self.artifacts_path = Path(artifacts_path)
        self.models = {}
        self.scalers = {}
        self.encoders = {}
        self.test_results = {}
    
    def load_all_models(self):
        """Load all available models from artifacts"""
        print("🚀 LOADING ALL MODELS FOR EVALUATION")
        print("=" * 60)
        
        # Try to load behavioral model
        try:
            behavioral_files = self._find_model_files('behavioral')
            if behavioral_files:
                self.models['behavioral'] = self._load_model_set(behavioral_files, 'behavioral')
                print("✅ Behavioral Fraud Model loaded")
        except Exception as e:
            print(f"❌ Failed to load behavioral model: {e}")
        
        # Try to load financial model
        try:
            financial_files = self._find_model_files('financial')
            if financial_files:
                self.models['financial'] = self._load_model_set(financial_files, 'financial')
                print("✅ Financial Credit Risk Model loaded")
        except Exception as e:
            print(f"❌ Failed to load financial model: {e}")
        
        # Try to load security model
        try:
            security_files = self._find_model_files('security')
            if security_files:
                self.models['security'] = self._load_model_set(security_files, 'security')
                print("✅ Network Security Model loaded")
        except Exception as e:
            print(f"❌ Failed to load security model: {e}")
        
        # Try to load your existing advanced model
        try:
            lgbm_path = self.artifacts_path / "models" / "lgbm_model.txt"
            if lgbm_path.exists():
                self.models['advanced_network'] = lgb.Booster(model_file=str(lgbm_path))
                print("✅ Advanced Network Model (lgbm_model.txt) loaded")
        except Exception as e:
            print(f"❌ Failed to load advanced model: {e}")
        
        print(f"\n📊 Total models loaded: {len(self.models)}")
        return len(self.models) > 0
    
    def _find_model_files(self, model_type):
        """Find model files for a specific type"""
        files = {}
        
        # Search patterns for different model files
        patterns = {
            'behavioral': ['behavioral_model', 'behavioral_scaler', 'behavioral_features', 'behavioral_encoders'],
            'financial': ['financial_model', 'financial_scaler', 'financial_features', 'financial_encoders'],
            'security': ['security_model', 'security_scaler', 'security_features', 'security_encoders']
        }
        
        if model_type not in patterns:
            return None
        
        for root, dirs, file_names in os.walk(self.artifacts_path):
            root_path = Path(root)
            
            for pattern in patterns[model_type]:
                for file_name in file_names:
                    if pattern in file_name.lower():
                        files[pattern] = root_path / file_name
        
        return files if files else None
    
    def _load_model_set(self, files, model_type):
        """Load complete model set (model + preprocessing)"""
        model_set = {}
        
        # Load main model
        model_file = files.get(f'{model_type}_model')
        if model_file and model_file.exists():
            if model_file.suffix == '.txt':
                model_set['model'] = lgb.Booster(model_file=str(model_file))
            else:
                model_set['model'] = joblib.load(model_file)
        
        # Load scaler
        scaler_file = files.get(f'{model_type}_scaler')
        if scaler_file and scaler_file.exists():
            model_set['scaler'] = joblib.load(scaler_file)
        
        # Load feature names
        features_file = files.get(f'{model_type}_features')
        if features_file and features_file.exists():
            model_set['features'] = joblib.load(features_file)
        
        # Load encoders
        encoders_file = files.get(f'{model_type}_encoders')
        if encoders_file and encoders_file.exists():
            model_set['encoders'] = joblib.load(encoders_file)
        
        return model_set
    
    def prepare_test_data(self):
        """Prepare test datasets for each model type"""
        print("\n📊 PREPARING TEST DATASETS")
        print("=" * 40)
        
        test_datasets = {}
        
        # Load Nigerian dataset for behavioral testing
        try:
            nigerian_path = self.artifacts_path / "raw" / "nigerian_sample.parquet"
            if nigerian_path.exists():
                df = pd.read_parquet(nigerian_path)
                df = self._clean_data(df)
                if 'is_fraud' not in df.columns:
                    df['is_fraud'] = self._create_synthetic_labels(df)
                test_datasets['behavioral'] = df.sample(n=min(5000, len(df)), random_state=42)
                print(f"✅ Behavioral test data: {len(test_datasets['behavioral'])} samples")
        except Exception as e:
            print(f"❌ Failed to load behavioral test data: {e}")
        
        # Load PaySim dataset for financial testing
        try:
            paysim_path = self.artifacts_path / "raw" / "paysim.parquet"
            if paysim_path.exists():
                df = pd.read_parquet(paysim_path)
                df = self._clean_data(df)
                if 'isFraud' in df.columns:
                    df['is_fraud'] = df['isFraud']
                elif 'is_fraud' not in df.columns:
                    df['is_fraud'] = self._create_synthetic_labels(df)
                test_datasets['financial'] = df.sample(n=min(5000, len(df)), random_state=42)
                print(f"✅ Financial test data: {len(test_datasets['financial'])} samples")
        except Exception as e:
            print(f"❌ Failed to load financial test data: {e}")
        
        # Load CIFER dataset for security testing
        try:
            cifer_path = self.artifacts_path / "raw" / "cifer_sample.parquet"
            if cifer_path.exists():
                df = pd.read_parquet(cifer_path)
                df = self._clean_data(df)
                if 'is_fraud' not in df.columns:
                    df['is_fraud'] = self._create_synthetic_labels(df)
                test_datasets['security'] = df.sample(n=min(5000, len(df)), random_state=42)
                print(f"✅ Security test data: {len(test_datasets['security'])} samples")
        except Exception as e:
            print(f"❌ Failed to load security test data: {e}")
        
        return test_datasets
    
    def _clean_data(self, df):
        """Clean dataset for evaluation"""
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            if df[col].notna().sum() > 0:
                cap_value = df[col].quantile(0.999)
                df[col] = df[col].clip(upper=cap_value)
        return df.fillna(0)
    
    def _create_synthetic_labels(self, df):
        """Create synthetic fraud labels for evaluation"""
        # Create labels based on suspicious patterns
        suspicious = (
            (df.get('amount', df.get('amount_ngn', 0)) > df.get('amount', df.get('amount_ngn', 0)).quantile(0.95)) |
            (df.get('velocity_score', 5) > 8) |
            (df.get('spending_deviation_score', 0.5) > 0.8) |
            (df.get('is_night_txn', False) == True)
        )
        return suspicious.astype(int)
    
    def evaluate_model(self, model_name, model_set, test_data):
        """Evaluate a single model comprehensively"""
        print(f"\n🎯 EVALUATING {model_name.upper()} MODEL")
        print("=" * 50)
        
        try:
            # Prepare features based on model type
            if model_name == 'behavioral':
                X, y = self._prepare_behavioral_features(test_data)
            elif model_name == 'financial':
                X, y = self._prepare_financial_features(test_data)
            elif model_name == 'security':
                X, y = self._prepare_security_features(test_data)
            else:
                # For advanced network model, use basic features
                feature_cols = ['amount', 'step', 'oldbalanceOrg', 'newbalanceOrig']
                available_cols = [c for c in feature_cols if c in test_data.columns]
                X = test_data[available_cols] if available_cols else test_data.select_dtypes(include=[np.number]).iloc[:, :10]
                y = test_data['is_fraud'] if 'is_fraud' in test_data.columns else test_data['isFraud']
            
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=42, stratify=y
            )
            
            # Preprocess if scaler available
            if 'scaler' in model_set and model_set['scaler'] is not None:
                X_test_scaled = model_set['scaler'].transform(X_test)
            else:
                X_test_scaled = X_test.values if hasattr(X_test, 'values') else X_test
            
            # Get predictions
            model = model_set['model'] if 'model' in model_set else model_set
            
            if hasattr(model, 'predict_proba'):
                y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
                y_pred = (y_pred_proba > 0.5).astype(int)
            elif hasattr(model, 'predict'):
                if str(type(model)).find('lightgbm') != -1:
                    y_pred_proba = model.predict(X_test_scaled)
                    y_pred = (y_pred_proba > 0.5).astype(int)
                else:
                    y_pred_proba = model.predict(X_test_scaled)
                    y_pred = (y_pred_proba > 0.5).astype(int)
            else:
                raise ValueError("Model doesn't have predict method")
            
            # Calculate metrics
            results = self._calculate_metrics(y_test, y_pred, y_pred_proba, model_name)
            
            # Store results
            self.test_results[model_name] = results
            
            # Print results
            self._print_results(model_name, results)
            
            return results
            
        except Exception as e:
            print(f"❌ Error evaluating {model_name}: {e}")
            return None
    
    def _prepare_behavioral_features(self, df):
        """Prepare behavioral features for evaluation"""
        # Simple behavioral features that should exist
        features = []
        
        # Device and behavioral features
        if 'device_seen_count' in df.columns:
            features.append(1 / (df['device_seen_count'] + 1))  # device_trust_score
        else:
            features.append(np.random.uniform(0.1, 1.0, len(df)))
        
        if 'is_device_shared' in df.columns:
            features.append((~df['is_device_shared']).astype(int))  # device_exclusivity
        else:
            features.append(np.random.binomial(1, 0.7, len(df)))
        
        if 'velocity_score' in df.columns:
            features.append(np.clip(df['velocity_score'] / 10, 0, 1))  # velocity_risk
        else:
            features.append(np.random.uniform(0, 1, len(df)))
        
        if 'spending_deviation_score' in df.columns:
            features.append(np.clip(df['spending_deviation_score'], 0, 1))  # spending_anomaly
        else:
            features.append(np.random.uniform(0, 1, len(df)))
        
        # Add basic amount features
        amount_col = 'amount_ngn' if 'amount_ngn' in df.columns else 'amount'
        if amount_col in df.columns:
            features.append(np.log1p(df[amount_col]))
            features.append((df[amount_col] % 1000 == 0).astype(int))
        else:
            features.append(np.random.uniform(0, 10, len(df)))
            features.append(np.random.binomial(1, 0.3, len(df)))
        
        X = pd.DataFrame(np.column_stack(features), columns=[
            'device_trust_score', 'device_exclusivity', 'velocity_risk', 
            'spending_anomaly', 'amount_log', 'round_amount'
        ])
        
        y = df['is_fraud'] if 'is_fraud' in df.columns else df.get('isFraud', np.random.binomial(1, 0.1, len(df)))
        
        return X, y
    
    def _prepare_financial_features(self, df):
        """Prepare financial features for evaluation"""
        # Basic financial features that should exist in PaySim
        feature_cols = [
            'amount', 'oldbalanceOrg', 'newbalanceOrig', 
            'oldbalanceDest', 'newbalanceDest', 'step'
        ]
        
        # Keep only existing columns
        available_cols = [col for col in feature_cols if col in df.columns]
        
        if available_cols:
            X = df[available_cols].copy()
            
            # Add engineered features
            if 'amount' in X.columns:
                X['amount_log'] = np.log1p(X['amount'])
                X['is_large_amount'] = (X['amount'] > X['amount'].quantile(0.95)).astype(int)
            
            if 'oldbalanceOrg' in X.columns and 'amount' in X.columns:
                X['balance_ratio'] = X['amount'] / (X['oldbalanceOrg'] + 1)
            
            if 'type' in df.columns:
                # Encode transaction type
                type_map = {'PAYMENT': 0, 'TRANSFER': 1, 'CASH_OUT': 2, 'DEBIT': 3, 'CASH_IN': 4}
                X['type_encoded'] = df['type'].map(type_map).fillna(0)
        else:
            # Fallback to numeric columns
            X = df.select_dtypes(include=[np.number]).iloc[:, :8]
        
        y = df['is_fraud'] if 'is_fraud' in df.columns else df.get('isFraud', np.random.binomial(1, 0.1, len(df)))
        
        return X, y
    
    def _prepare_security_features(self, df):
        """Prepare security features for evaluation"""
        # Basic security-focused features
        features = []
        
        # Amount-based security features
        if 'amount' in df.columns:
            features.append(np.log1p(df['amount']))
            features.append((df['amount'] < 1).astype(int))  # micro transactions
            features.append((df['amount'] % 100 == 0).astype(int))  # round amounts
        else:
            features.extend([np.random.uniform(0, 10, len(df)), 
                           np.random.binomial(1, 0.1, len(df)),
                           np.random.binomial(1, 0.3, len(df))])
        
        # Time-based security features
        if 'step' in df.columns:
            features.append(df['step'] % 24)  # hour
            features.append(((df['step'] % 24) > 22).astype(int))  # night activity
        else:
            features.extend([np.random.uniform(0, 24, len(df)),
                           np.random.binomial(1, 0.2, len(df))])
        
        # Account-based security features
        if 'oldbalanceOrg' in df.columns:
            features.append((df['oldbalanceOrg'] == 0).astype(int))  # zero balance
        else:
            features.append(np.random.binomial(1, 0.15, len(df)))
        
        X = pd.DataFrame(np.column_stack(features), columns=[
            'amount_log', 'micro_transaction', 'round_amount', 
            'transaction_hour', 'night_activity', 'zero_balance'
        ])
        
        y = df['is_fraud'] if 'is_fraud' in df.columns else df.get('isFraud', np.random.binomial(1, 0.1, len(df)))
        
        return X, y
    
    def _calculate_metrics(self, y_true, y_pred, y_pred_proba, model_name):
        """Calculate comprehensive evaluation metrics"""
        results = {
            'model_name': model_name,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'auc_score': roc_auc_score(y_true, y_pred_proba) if len(set(y_true)) > 1 else 0.5,
            'fraud_rate': y_true.mean(),
            'prediction_rate': y_pred.mean(),
            'samples_tested': len(y_true)
        }
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            results.update({
                'true_negatives': int(tn),
                'false_positives': int(fp),
                'false_negatives': int(fn),
                'true_positives': int(tp)
            })
        
        return results
    
    def _print_results(self, model_name, results):
        """Print evaluation results"""
        print(f"\n📈 {model_name.upper()} MODEL PERFORMANCE:")
        print(f"   Samples tested: {results['samples_tested']:,}")
        print(f"   Actual fraud rate: {results['fraud_rate']:.3%}")
        print(f"   Predicted fraud rate: {results['prediction_rate']:.3%}")
        print(f"\n📊 Performance Metrics:")
        print(f"   AUC Score: {results['auc_score']:.4f}")
        print(f"   Accuracy: {results['accuracy']:.4f}")
        print(f"   Precision: {results['precision']:.4f}")
        print(f"   Recall: {results['recall']:.4f}")
        print(f"   F1-Score: {results['f1_score']:.4f}")
        
        if all(k in results for k in ['true_positives', 'false_positives', 'false_negatives', 'true_negatives']):
            print(f"\n🎯 Confusion Matrix:")
            print(f"   True Positives: {results['true_positives']}")
            print(f"   False Positives: {results['false_positives']}")
            print(f"   True Negatives: {results['true_negatives']}")
            print(f"   False Negatives: {results['false_negatives']}")
    
    def run_full_evaluation(self):
        """Run comprehensive evaluation of all models"""
        print("🚀 COMPREHENSIVE MODEL EVALUATION SYSTEM")
        print("=" * 70)
        
        # Load models
        models_loaded = self.load_all_models()
        if not models_loaded:
            print("❌ No models found for evaluation!")
            return
        
        # Prepare test datasets
        test_datasets = self.prepare_test_data()
        
        # Evaluate each model
        for model_name, model_set in self.models.items():
            if model_name in test_datasets:
                self.evaluate_model(model_name, model_set, test_datasets[model_name])
            elif 'advanced_network' in model_name and test_datasets:
                # Use any available dataset for advanced model
                dataset_name = list(test_datasets.keys())[0]
                self.evaluate_model(model_name, model_set, test_datasets[dataset_name])
        
        # Generate summary report
        self.generate_summary_report()
        
        return self.test_results
    
    def generate_summary_report(self):
        """Generate comprehensive summary report"""
        if not self.test_results:
            print("❌ No evaluation results available!")
            return
        
        print("\n🏆 EVALUATION SUMMARY REPORT")
        print("=" * 60)
        
        # Create comparison table
        comparison_data = []
        for model_name, results in self.test_results.items():
            comparison_data.append({
                'Model': model_name.replace('_', ' ').title(),
                'AUC': f"{results['auc_score']:.4f}",
                'Accuracy': f"{results['accuracy']:.4f}",
                'Precision': f"{results['precision']:.4f}",
                'Recall': f"{results['recall']:.4f}",
                'F1-Score': f"{results['f1_score']:.4f}",
                'Samples': f"{results['samples_tested']:,}"
            })
        
        comparison_df = pd.DataFrame(comparison_data)
        print("\n📊 MODEL COMPARISON TABLE:")
        print(comparison_df.to_string(index=False))
        
        # Find best performing model
        if len(self.test_results) > 1:
            best_auc = max(self.test_results.items(), key=lambda x: x[1]['auc_score'])
            best_f1 = max(self.test_results.items(), key=lambda x: x[1]['f1_score'])
            
            print(f"\n🏆 BEST PERFORMING MODELS:")
            print(f"   Best AUC: {best_auc[0]} ({best_auc[1]['auc_score']:.4f})")
            print(f"   Best F1: {best_f1[0]} ({best_f1[1]['f1_score']:.4f})")
        
        # Save results
        results_path = self.artifacts_path / "evaluation_results.json"
        with open(results_path, 'w') as f:
            json.dump(self.test_results, f, indent=2)
        
        print(f"\n💾 Results saved to: {results_path}")

def run_evaluation():
    """Main evaluation function"""
    evaluator = ModelEvaluator()
    results = evaluator.run_full_evaluation()
    return evaluator, results

if __name__ == "__main__":
    evaluator, results = run_evaluation()

🚀 COMPREHENSIVE MODEL EVALUATION SYSTEM
🚀 LOADING ALL MODELS FOR EVALUATION
✅ Behavioral Fraud Model loaded
✅ Financial Credit Risk Model loaded
✅ Network Security Model loaded

📊 Total models loaded: 3

📊 PREPARING TEST DATASETS
❌ No evaluation results available!
